### **감성분석(api)**
1. 원본 리뷰 데이터
2. 전처리
3. 감성분석(API)
4. 감성 라벨링
5. csv 저장
6. 사용목적 분석(카테고리 분류)
7. 감성 x 사용목적 결합 분석
8. 최종 인사이트 도출
#
1. 최초실행 -> 100개 -> csv저장
2. 이후 실행 -> 이어서 500개씩 -> csv에 append
- 500개씩 누적 처리 -> 기존 csv에 append
- 매 실행은 독립 실행 버튼
- 실행할때마다 
    - checkpoint 기준으로 이어서 시작
    - batch 만큼 처리
    - csv에 이어쓰기
    - checkpoint 갱신
### ***주의***: **checkpoint 절대 삭제하지 말 것. 삭제하면 0부터 시작해야 함**
#
#### **출력결과**
제품명|조회수|리뷰|sentiment_score|sentiment_magnitude|sentiment_label
#
### **실행 확인**
- ex) 처리 범위: 100 ~ 600
- ex) status code: 200
- 코드가 200이 나오면 정상 실행 
- status code: 403 -> 실패 -> API 활성화 되어있는지 확인하기

### **공통 설정 (한 번만 실행)**

In [ ]:
import pandas as pd
import requests
import time
import json
import os

# API 설정
API_KEY = "gen-lang-client-0540508630"
URL = f"https://language.googleapis.com/v1/documents:analyzeSentiment?key={API_KEY}"

# 파일 경로
INPUT_FILE = "naver_lg.csv"
OUTPUT_FILE = "감성분석결과.csv"
CHECKPOINT_FILE = "checkpoint.json"

### **함수 정의 (한 번만 실행)**

In [16]:
def get_label(score):
    if score is None:
        return None
    elif score >= 0.15:
        return "긍정"
    elif score <= -0.15:
        return "부정"
    else:
        return "중립"


def analyze_sentiment(text):
    data = {
        "document": {
            "type": "PLAIN_TEXT",
            "content": str(text)[:1000]
        },
        "encodingType": "UTF8"
    }

    res = requests.post(URL, json=data, timeout=10)

    print("status code:", res.status_code) # API 응답 확인, 200이면 정상

    if res.status_code == 200:
        result = res.json()
        score = result['documentSentiment']['score']
        magnitude = result['documentSentiment']['magnitude']
        return score, magnitude
    else:
        print("error:", res.text)
        return None, None


def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            return json.load(f).get("last_index", 0)
    return 0


def save_checkpoint(index):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"last_index": index}, f)

### **데이터 로드 및 전처리 (한 번만 실행)**

In [17]:
df = pd.read_csv(INPUT_FILE)

df['clean_review'] = (
    df['clean_review']
    .astype(str)
    .str.replace(r'[\r\n]+', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

### **실행 파라미터 설정 (매 실행 시 수정 없이 사용)**

In [18]:
FIRST_BATCH_SIZE = 100
BATCH_SIZE = 500

### ***실행 블록 (이 부분을 "버튼처럼" 반복 실행)***

In [30]:
start_idx = load_checkpoint()

# 최초 실행이면 100개, 이후는 500개
if start_idx == 0:
    batch_size = FIRST_BATCH_SIZE
else:
    batch_size = BATCH_SIZE

end_idx = start_idx + batch_size

df_batch = df.iloc[start_idx:end_idx].copy()

print(f"처리 범위: {start_idx} ~ {end_idx}")

# 결과 컬럼 초기화
df_batch['sentiment_score'] = None
df_batch['sentiment_magnitude'] = None
df_batch['sentiment_label'] = None

# 감성 분석 수행
for i in df_batch.index:
    text = df_batch.loc[i, 'clean_review']

    try:
        score, magnitude = analyze_sentiment(text)
    except Exception as e:
        print("분석 실패:", e)
        score, magnitude = None, None

    df_batch.loc[i, 'sentiment_score'] = score
    df_batch.loc[i, 'sentiment_magnitude'] = magnitude
    df_batch.loc[i, 'sentiment_label'] = get_label(score)

# CSV 저장 (append 구조)
if start_idx == 0:
    df_batch.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
else:
    df_batch.to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding='utf-8-sig')

# checkpoint 업데이트
save_checkpoint(end_idx)

print(f"완료: {start_idx} ~ {end_idx}")
print(f"다음 시작 index: {end_idx}")

처리 범위: 4100 ~ 4600
status code: 400
error: {
  "error": {
    "code": 400,
    "message": "API key not valid. Please pass a valid API key.",
    "status": "INVALID_ARGUMENT",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.ErrorInfo",
        "reason": "API_KEY_INVALID",
        "domain": "googleapis.com",
        "metadata": {
          "service": "language.googleapis.com"
        }
      },
      {
        "@type": "type.googleapis.com/google.rpc.LocalizedMessage",
        "locale": "en-US",
        "message": "API key not valid. Please pass a valid API key."
      }
    ]
  }
}

status code: 400
error: {
  "error": {
    "code": 400,
    "message": "API key not valid. Please pass a valid API key.",
    "status": "INVALID_ARGUMENT",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.ErrorInfo",
        "reason": "API_KEY_INVALID",
        "domain": "googleapis.com",
        "metadata": {
          "service": "language.googleapis.com"
 

KeyboardInterrupt: 

In [20]:
print(df.columns.tolist())

['product_id', 'product_name', 'price', 'option', 'review', 'review_len', 'product_tags', 'clean_review', 'tokens', 'option_clean']
